# 02 · Harvesting and preparing the comments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eabanoz/bocelli-nostalgia/blob/main/notebooks/02_harvest_and_prepare.ipynb)

**Target:** `TdWEhMOrRpQ` — *Con Te Partirò / Time To Say Goodbye
(Live From Piazza Dei Cavalieri, Italy / 1997)*

| | |
|---|---|
| Performed | **1997** |
| Uploaded | **23 October 2015** |
| Comments | 26,936 |

> **You can run this without an API key.** The harvested dataset is
> committed to the repo. Set `HARVEST = False` (the default) to load it and
> skip straight to preparation.

---

## The three clocks

This video is an archival re-release, and that is the analytic centre of the
study rather than a footnote. Every comment carries three timestamps:

1. **Performance time** — 1997, fixed
2. **Upload time** — October 2015, fixed
3. **Comment time** — 2015 to now, and it *varies*

A comment written in 2024 sits 27 years after the performance and 9 years
after the upload. Nostalgia is a claim about distance from a past, and here
we can measure that distance for every utterance. Most YouTube research
collapses these into one date and loses the structure.

## Why `order="time"`

`relevance` returns roughly the most-liked comments first; `time` returns
chronological. We use **`time`**, because the question is about change over
the comment period, and relevance-ranking would distort the temporal
distribution we intend to analyse. The trade-off: we cannot say anything
about which comments the community elevated.

In [ ]:
# --- Clone the repo and set the working directory -------------------------
import sys, os, subprocess
from pathlib import Path

REPO = "bocelli-nostalgia"
if "google.colab" in sys.modules:
    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/eabanoz/bocelli-nostalgia.git"], check=True)
    ROOT = Path(REPO).resolve()
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
for d in ["data/raw", "data/processed", "outputs/figures"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)

In [ ]:
import pandas as pd, numpy as np, re, html, hashlib, time
import matplotlib.pyplot as plt

pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.dpi"] = 110

VIDEO_ID  = "TdWEhMOrRpQ"
PERF_YEAR = 1997
UPLOAD_DT = pd.Timestamp("2015-10-23", tz="UTC")

HARVEST = False   # True to re-collect from the API (needs a key, ~270 units)

## 1 · Harvest (or load)

About 270 quota units for ~27,000 top-level threads. Replies come free
inside the same response.

**Expect fewer comments than the displayed count.** The API caps pagination
on large videos and excludes deleted or held comments. We retrieved 18,923
top-level comments of 26,936 displayed — 70.3%. Report that as a large
sample, not a census.

In [ ]:
RAW = Path("data/raw/comments_raw.csv")

if HARVEST:
    API_KEY = None
    try:
        from google.colab import userdata
        API_KEY = userdata.get("YOUTUBE_API_KEY")
    except Exception:
        API_KEY = os.environ.get("YOUTUBE_API_KEY")
    if not API_KEY:
        from getpass import getpass
        API_KEY = getpass("YouTube API key (hidden): ")

    from src.channel_client import ChannelCollector
    cc = ChannelCollector(API_KEY)
    t0 = time.time()
    comments = cc.comments(VIDEO_ID, max_pages=400, order="time",
                           include_replies=True)
    print(f"Retrieved {len(comments):,} in {time.time()-t0:.0f}s | "
          f"{cc.quota_report()}")
    comments.to_csv(RAW, index=False)     # gitignored: holds author IDs
else:
    src = Path("data/processed/comments_clean.csv")
    comments = pd.read_csv(src)
    print(f"Loaded prepared dataset: {len(comments):,} comments")
    print("Set HARVEST = True to collect from the API instead.")

print(f"  top-level : {(~comments.is_reply).sum():,}")
print(f"  replies   : {comments.is_reply.sum():,}")

## 2 · Pseudonymise immediately

Not at publication time — now. `author_channel_id` identifies a natural
person, and these comments describe dead relatives, weddings and funerals.
Public platform, sensitive material.

A plain SHA-256 of a channel ID is **not** pseudonymisation: channel IDs come
from an enumerable space, so an unsalted hash is invertible by anyone with
the same function. The salt is the whole security property, and it must not
travel with the data.

In [ ]:
if HARVEST and "author_channel_id" in comments.columns:
    import secrets
    SALT = None
    try:
        from google.colab import userdata
        SALT = userdata.get("HASH_SALT")
    except Exception:
        SALT = os.environ.get("HASH_SALT")
    if not SALT:
        SALT = secrets.token_hex(16)
        print(f"Generated ephemeral salt: {SALT}")
        print("Store as Colab secret HASH_SALT for stable pseudonyms.")

    comments["author_pseudo"] = comments.author_channel_id.map(
        lambda x: hashlib.sha256((SALT + str(x)).encode()).hexdigest()[:16]
        if pd.notna(x) and str(x) else "")
    comments = comments.drop(columns=["author_channel_id"])

print(f"{comments.author_pseudo.nunique():,} unique commenters across "
      f"{len(comments):,} comments")
print(f"Mean comments per person: "
      f"{len(comments)/comments.author_pseudo.nunique():.2f}")
print("Almost no repeat commenters — a broad population, not a conversation.")

## 3 · Text cleaning — deliberately light

Strip URLs, @mentions and timestamps. **Keep** casing, punctuation and emoji:
transformer models were pretrained on raw web text and use all three.
Lowercasing and stopword removal are bag-of-words habits that destroy signal.

One nostalgia-specific decision: we do **not** drop short comments. In most
corpora `"2026?"` is noise. Here it is a temporal-positioning ritual — a
specific and highly relevant speech act. Filtering by length would delete
the phenomenon.

In [ ]:
URL_RE    = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@[\w\-.]+")
TS_RE     = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")
WS_RE     = re.compile(r"\s+")
EMOJI_RE  = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF"
                       "\U0001F000-\U0001F0FF\U00002190-\U000021FF"
                       "\U00002B00-\U00002BFF]+")
REPEAT_RE = re.compile(r"(.)\1{3,}")

def clean(s):
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)
    s = URL_RE.sub(" ", s); s = MENTION_RE.sub(" ", s); s = TS_RE.sub(" ", s)
    s = REPEAT_RE.sub(r"\1\1\1", s)
    return WS_RE.sub(" ", s).strip()

if "text_clean" not in comments.columns or HARVEST:
    comments["text_clean"] = comments.text.map(clean)
    comments["n_emoji"] = comments.text.map(lambda s: len(EMOJI_RE.findall(s or "")))
    comments["n_chars"] = comments.text_clean.str.len()
    comments["n_tokens"] = comments.text_clean.str.split().str.len().fillna(0)
    comments["is_textless"] = comments.text_clean.str.replace(
        EMOJI_RE, "", regex=True).str.strip().str.len() < 2

print(f"Empty / emoji-only : {comments.is_textless.sum():,} "
      f"({100*comments.is_textless.mean():.1f}%)")
print(f"Median length      : {comments.n_tokens.median():.0f} tokens")
print(f"With emoji         : {100*(comments.n_emoji>0).mean():.1f}%")

## 4 · Language detection

Bocelli's audience is global, so we need the distribution before choosing a
modelling strategy.

**lingua** outperforms `langdetect` on short text, which matters when the
median comment is five tokens. We restrict it to a plausible candidate set —
an unrestricted detector will confidently assign Xhosa to a three-word
English comment. Detection on very short strings remains unreliable, so we
keep the confidence value and treat low-confidence assignments as their own
category rather than pretending they are known.

In [ ]:
if HARVEST or "lang" not in comments.columns:
    try:
        from lingua import Language, LanguageDetectorBuilder
    except ImportError:
        !pip install -q lingua-language-detector
        from lingua import Language, LanguageDetectorBuilder

    LANGS = [Language.ITALIAN, Language.ENGLISH, Language.SPANISH,
             Language.PORTUGUESE, Language.FRENCH, Language.GERMAN,
             Language.POLISH, Language.ROMANIAN, Language.DUTCH,
             Language.TURKISH, Language.RUSSIAN, Language.INDONESIAN,
             Language.JAPANESE, Language.KOREAN, Language.CHINESE,
             Language.ARABIC, Language.GREEK, Language.CZECH,
             Language.HUNGARIAN, Language.SWEDISH, Language.VIETNAMESE,
             Language.TAGALOG, Language.HINDI]
    det = (LanguageDetectorBuilder.from_languages(*LANGS)
           .with_preloaded_language_models().build())

    langs, confs = [], []
    for i, t in enumerate(comments.text_clean.fillna("")):
        if len(t.strip()) < 3:
            langs.append(None); confs.append(0.0); continue
        v = det.compute_language_confidence_values(t)
        langs.append(v[0].language.iso_code_639_1.name.lower() if v else None)
        confs.append(float(v[0].value) if v else 0.0)
        if i and i % 5000 == 0:
            print(f"   {i:,}/{len(comments):,}")
    comments["lang"] = langs
    comments["lang_conf"] = confs

comments["lang_reliable"] = (comments.lang.notna() &
                             (comments.lang_conf >= 0.55) &
                             (comments.n_tokens >= 3))
print(f"Reliable: {comments.lang_reliable.sum():,} of {len(comments):,} "
      f"({100*comments.lang_reliable.mean():.1f}%)")

### The distribution, and its fragility

The shares depend heavily on where you set the length and confidence
thresholds. Show that before quoting any single number.

In [ ]:
rows = []
for tok in [0, 3, 5, 8, 12]:
    for conf in [0.0, 0.55, 0.70, 0.85]:
        m = (comments.lang.notna() & ~comments.is_textless &
             (comments.n_tokens >= tok) & (comments.lang_conf >= conf))
        s = comments[m].lang.value_counts(normalize=True).mul(100)
        rows.append({"min_tok": tok, "min_conf": conf, "n": int(m.sum()),
                     **{l: round(s.get(l, 0), 1)
                        for l in ["pt", "en", "it", "es", "fr", "ru"]}})
sens = pd.DataFrame(rows)
display(sens)

print("English rises with the length threshold, falls with the confidence")
print("threshold; Portuguese does the reverse. Which language 'leads' is")
print("set by cell parameters.\n")
print("Mean detection confidence by language:")
print(comments[comments.lang.notna()].groupby("lang").lang_conf
        .agg(["mean", "count"]).sort_values("count", ascending=False)
        .head(8).round(3).to_string())
print("\nEnglish has the LOWEST confidence of any major language: short,")
print("ambiguous Latin-script text defaults to it.")

In [ ]:
sub = comments[comments.lang.notna() & ~comments.is_textless &
               (comments.n_tokens >= 5) & (comments.lang_conf >= 0.55)]
dist = sub.lang.value_counts(normalize=True).mul(100).round(1)
cnt = sub.lang.value_counts()

print(f"SUBSTANTIVE COMMENTS (>=5 tokens, conf>=0.55): {len(sub):,}\n")
for l in dist.head(10).index:
    print(f"  {l:5} {dist[l]:5.1f}%   n={cnt[l]:,}")
print("\nEnglish and Portuguese are CO-DOMINANT at ~35% each. Italian is")
print("8.5%. There is no defensible way to call this an Italian corpus.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
dist.head(12).plot.barh(ax=ax[0], color="#4C72B0"); ax[0].invert_yaxis()
ax[0].set_title("Language share (%)")
sub.lang_conf.hist(bins=40, ax=ax[1], color="#55A868")
ax[1].set_title("Detection confidence")
plt.tight_layout()
plt.savefig("outputs/figures/languages.png", bbox_inches="tight")
plt.show()

## 5 · The three clocks

In [ ]:
comments["published_at"] = pd.to_datetime(comments.published_at, utc=True,
                                          errors="coerce")
comments["comment_year"] = comments.published_at.dt.year
comments["comment_ym"] = (comments.published_at.dt.to_period("M")
                          .dt.to_timestamp())
comments["years_since_performance"] = comments.comment_year - PERF_YEAR
comments["days_since_upload"] = (comments.published_at - UPLOAD_DT).dt.days
comments["years_since_upload"] = comments.days_since_upload / 365.25

print("Comment period:", comments.published_at.min().date(), "->",
      comments.published_at.max().date())
print(f"Distance from performance: "
      f"{comments.years_since_performance.min():.0f}–"
      f"{comments.years_since_performance.max():.0f} years\n")
print(comments.comment_year.value_counts().sort_index().to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
comments.groupby("comment_ym").size().plot(ax=ax[0], color="#4C72B0")
ax[0].set_title("Comment volume over time"); ax[0].set_xlabel("")

r = comments[comments.lang_reliable]
ly = pd.crosstab(r.comment_year, r.lang, normalize="index").mul(100)
ly[[c for c in ["en","pt","it","es","fr"] if c in ly.columns]].plot(
    ax=ax[1], marker="o")
ax[1].set_title("Language mix by year (%)"); ax[1].set_xlabel("")
ax[1].legend(fontsize=8, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig("outputs/figures/comments_over_time.png", bbox_inches="tight")
plt.show()

print("A shifting language share is a fact about who the algorithm served")
print("this video to over time — not about the song.")

## 6 · A first look before modelling

Read some comments. This is how you discover that theoretical categories do
not fit the data, while there is still time to change them.

In [ ]:
PROBES = {
 "who_listening": r"(?i)(?:who(?:'?s| is)? (?:still )?(?:listening|watching)|"
                  r"chi (?:ascolta|guarda)|qui[eé]n (?:escucha|sigue)|"
                  r"quem (?:est[aá] ouvindo|ouve)|\b20[12]\d\s*\?)",
 "personal_memory": r"(?i)(?:my (?:mother|mom|father|dad|grandma|grandmother)|"
                    r"mia (?:madre|mamma|nonna)|minha (?:mãe|avó)|"
                    r"meu (?:pai|avô)|when i was|quando ero|quando eu era)",
 "mortality": r"(?i)(?:rest in peace|\brip\b|passed away|funeral|"
              r"riposa in pace|descanse em paz|faleceu|saudades|"
              r"repose en paix|décédé)",
}
for name, pat in PROBES.items():
    hit = comments.text_clean.fillna("").str.contains(pat, regex=True)
    comments[f"probe_{name}"] = hit
    print(f"{name:18} {hit.sum():>6,}  ({100*hit.mean():.2f}%)")

for name in PROBES:
    s = comments[comments[f"probe_{name}"]]
    print(f"\n--- {name.upper()} ---")
    for t in s.text_clean.sample(min(3, len(s)), random_state=7):
        print("  •", str(t)[:120])

In [ ]:
comments.to_csv("data/processed/comments_clean.csv", index=False)
sens.to_csv("outputs/language_sensitivity.csv", index=False)
print(f"Saved {len(comments):,} comments to data/processed/")

In [ ]:
# --- Download the outputs (optional) --------------------------------------
import shutil
shutil.make_archive("/content/bocelli_step02", "zip", ROOT / "data/processed")
try:
    from google.colab import files
    files.download("/content/bocelli_step02.zip")
except Exception:
    print("Not in Colab — files are in data/processed/")

---

## What to report

- 21,469 comments retrieved; 18,923 top-level = 70.3% of the displayed count
- 19,063 unique authors — a broad population, not a conversation
- 46.7% reliable language detection (median comment is 5 tokens)
- English and Portuguese co-dominant at ~35%; Italian 8.5%
- Language shares move ten points across reasonable threshold choices

**Next:** [`03_nostalgia_analysis.ipynb`](03_nostalgia_analysis.ipynb)